# 05 — Content-Based Filtering
**Event Recommendation System** · Stage 6

Recommend events that are **similar to what the user already likes**, using the event vectors from notebook 04 and a profile vector for each user.

## The idea
1. Every event is a vector (TF-IDF text + category + city + price).
2. Every user gets a **profile vector** in the same space: a weighted average of the events they engaged with, mixed with their stated interests and home city.
3. Score each upcoming event by its **cosine similarity** to the profile and show the top 10.

Strength: it can recommend a **brand-new event** the moment it is listed (it only needs the description). Weakness: it only recommends "more of the same" and cannot learn hidden tastes that are not written in the text.

### Setup (run this first)
- **In Google Colab** this cell downloads the project from GitHub (data + `src/` code) and moves into it.
- **On your own computer** it just moves from `notebooks/` to the project root.

`sys.path.insert(0, "src")` lets us `import recsys` - the small package that holds the reusable code (cleaning, models, metrics). Notebooks *explain* the code; `src/recsys` *stores* it so every notebook uses exactly the same implementation.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/Abdelrahman2115/recommender-system.git"

if "google.colab" in sys.modules:                       # running inside Google Colab
    if not os.path.exists("/content/recommender-system"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/recommender-system"], check=True)
    os.chdir("/content/recommender-system")
elif os.path.basename(os.getcwd()) == "notebooks":      # running locally from the notebooks folder
    os.chdir("..")

sys.path.insert(0, "src")                               # make `import recsys` work
print("Project root:", os.getcwd())

Project root: /home/claude/recommender-system


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize

from recsys.data import load_clean, temporal_split, TRAIN_END, VAL_END
from recsys.features import EventFeaturizer
from recsys.models import ContentBasedRecommender, RandomRecommender, PopularityRecommender, GlobalMeanPredictor, PROFILE_WEIGHT
from recsys.evaluation import make_ranking_task, evaluate_ranking, evaluate_rating, recommend
from recsys.params import update_params
from recsys.plotting import set_style, bar_labels, MODEL_COLORS
set_style()

users, events, inter = load_clean()
train, val, test = temporal_split(inter)        # in this notebook: fit on TRAIN, evaluate on VALIDATION

## 1. Building one user's profile, step by step

### 1.1 How much does each interaction count?
| Interaction | Weight | Reason |
|---|---|---|
| view | 0.5 | weak signal: just looked |
| interested | 1.0 | clicked "interested" |
| going | 2.0 | committed |
| attended | rating − 2 | 1★ → −1 (moves the profile *away* from this kind of event) … 5★ → +3 |

Using the rating lets a bad experience count *against* similar events — a plain average would treat every attended event as liked.

In [3]:
featurizer = EventFeaturizer()
X = featurizer.fit_transform(events).toarray()              # (800 events, n features), unit-length rows
row_of = pd.Series(np.arange(len(events)), index=events["event_id"])

uid = train["user_id"].value_counts().index[50]             # a typical, fairly active user
hist = train[train["user_id"] == uid].copy()
hist["w"] = hist["interaction_type"].map(PROFILE_WEIGHT)
hist.loc[hist["interaction_type"] == "attended", "w"] = hist["rating"] - 2
hist = hist.merge(events[["event_id", "title", "category", "city"]], on="event_id")
print("user", uid, "| stated interests:", users.set_index("user_id").loc[uid, "interests"],
      "| home city:", users.set_index("user_id").loc[uid, "city"])
hist[["title", "category", "city", "interaction_type", "rating", "w"]].head(10)

user 744 | stated interests: Business & Career|Music|Outdoors & Travel | home city: Luxor


,title,category,city,interaction_type,rating,w
0,Interview Skills Session,Business & Career,Cairo,view,NaN,0.5
1,Oud Masterclass,Music,Cairo,attended,5.0,3.0
2,Pitching Night,Business & Career,Alexandria,attended,4.0,2.0
3,Social Media Gathering,Business & Career,Cairo,attended,5.0,3.0
4,Classical Arabic Gathering,Music,Hurghada,view,NaN,0.5
5,Astronomy Session,Education & Workshops,Cairo,interested,NaN,1.0
6,Theatre Gathering,Arts & Culture,Mansoura,view,NaN,0.5
7,Tarab Session,Music,Giza,interested,NaN,1.0
8,Pitching Session,Business & Career,Cairo,attended,4.0,2.0
9,Rock Night,Music,Hurghada,view,NaN,0.5


### 1.2 Behaviour profile
`behaviour = Σ weight × event_vector` — a matrix multiplication of the weight vector with the event vectors.

In [4]:
behaviour = hist["w"].to_numpy() @ X[row_of[hist["event_id"]].to_numpy()]
n_hist = len(hist)
print("behaviour profile length:", behaviour.shape, "| interactions used:", n_hist)

behaviour profile length: (2313,) | interactions used: 46


### 1.3 Interest profile (what the user said at sign-up)
- For each category we compute its **centroid** = the average vector of all events in that category.
- The interest profile = average centroid of the user's stated categories **+ 0.5 × their home-city vector**.

In [5]:
cats = events["category"].to_numpy()
centroids = {c: normalize(X[cats == c].mean(axis=0, keepdims=True))[0] for c in np.unique(cats)}

user_row = users.set_index("user_id").loc[uid]
cent = [centroids[c] for c in user_row["interests"].split("|")]
interest = normalize(np.mean(cent, axis=0, keepdims=True))[0] + 0.5 * featurizer.city_vector(user_row["city"])

### 1.4 Blend: history takes over as it grows
`profile = interest × s/(n+s) + behaviour × n/(n+s)` with `s = 5` (both parts normalised first).

- New user (n = 0): 100% interests.
- 5 interactions: 50/50.
- 45 interactions: 90% behaviour.

This is a **shrinkage** formula: trust the data in proportion to how much of it there is.

In [6]:
s = 5.0
profile = normalize(interest[None])[0] * s / (n_hist + s) + normalize(behaviour[None])[0] * n_hist / (n_hist + s)

# Same thing computed by the project's class -> must match exactly
cbf = ContentBasedRecommender(featurizer=EventFeaturizer(), interest_strength=5.0, city_strength=0.5).fit(train, users, events)
print("matches recsys.ContentBasedRecommender:", np.allclose(profile, cbf.profile(uid)))

vocab = featurizer.tfidf.get_feature_names_out()
text_part = profile[:len(vocab)]
print("strongest words in this user's profile:", list(vocab[np.argsort(-text_part)[:8]]))

matches recsys.ContentBasedRecommender: True
strongest words in this user's profile: ['career', 'marketing', 'business', 'business career', 'interview skills', 'interview', 'skills', 'networking']


## 2. Recommending
At the validation cut-off (1 Jul 2025) we rank the events that take place in the next two months, skipping events the user already interacted with. `recommend()` sorts by score and keeps the top 10.

In [7]:
task = make_ranking_task(train, val, events, TRAIN_END, VAL_END)
print(f"candidate events: {len(task.candidates)} | users to evaluate: {len(task.relevant)}")

top10 = recommend(cbf, uid, task.candidates, task.seen.get(uid, set()), k=10)
recs = events.set_index("event_id").loc[top10, ["title", "category", "city", "price"]]
recs["score"] = cbf.score(uid, top10).round(3)
recs["went? (validation)"] = ["✔" if e in task.relevant.get(uid, set()) else "" for e in top10]
recs

candidate events: 75 | users to evaluate: 1081


,title,category,city,price,score,went? (validation)
event_id,,,,,,
667,Networking Meetup,Business & Career,Cairo,0.0,0.712,
692,Intro To Interview Skills,Business & Career,Cairo,0.0,0.712,✔
254,Content Workshop,Business & Career,Cairo,0.0,0.702,
231,Content Session,Business & Career,Cairo,0.0,0.697,
408,Startup Masterclass,Business & Career,Cairo,230.0,0.645,
15,Social Media Session,Business & Career,Hurghada,140.0,0.439,
317,Stage Workshop,Music,Cairo,0.0,0.435,
448,Techno Night,Music,Cairo,0.0,0.432,
32,Dj Workshop,Music,Cairo,230.0,0.410,


## 3. Evaluation on the validation set
Metrics are explained in detail in notebook 08. In short:
- **Precision@10** — of the 10 events shown, how many did the user go to?
- **Recall@10** — of the events the user went to, how many were in our 10?
- **NDCG@10** — like recall, but a hit at position 1 counts more than a hit at position 10.

We always compare against two baselines: **random** and **popularity**.

In [8]:
models = {"Random": RandomRecommender(), "Popularity": PopularityRecommender().fit(train), "Content-Based": cbf}
results = pd.DataFrame({name: evaluate_ranking(m, task).mean(numeric_only=True) for name, m in models.items()}).T
results = results[["precision@10", "recall@10", "ndcg@10"]].round(4)
results

,precision@10,recall@10,ndcg@10
Random,0.0253,0.1236,0.0637
Popularity,0.0379,0.2010,0.1379
Content-Based,0.1329,0.6998,0.4272


## 4. Tuning (on validation, never on test)
Two groups of settings:
1. **Profile:** `interest_strength` (s) and `city_strength`.
2. **Feature weights** of the category and city blocks.

We pick the combination with the best **NDCG@10** on the validation period.

In [9]:
import itertools

rows = []
for s_, city_ in itertools.product([2, 5, 15], [0.0, 0.5, 1.0]):
    m = ContentBasedRecommender(interest_strength=s_, city_strength=city_).fit(train, users, events)
    r = evaluate_ranking(m, task).mean(numeric_only=True)
    rows.append({"interest_strength": s_, "city_strength": city_, "recall@10": r["recall@10"], "ndcg@10": r["ndcg@10"]})
grid1 = pd.DataFrame(rows).sort_values("ndcg@10", ascending=False)
best1 = grid1.iloc[0]
grid1.round(4).head(5)

,interest_strength,city_strength,recall@10,ndcg@10
5,5,1.0,0.6985,0.4354
8,15,1.0,0.6890,0.4317
2,2,1.0,0.6917,0.4298
1,2,0.5,0.6973,0.4284
4,5,0.5,0.6998,0.4272


In [10]:
rows = []
for cat_w, city_w in [(0.25, 0.5), (0.5, 0.25), (0.5, 0.5), (1.0, 0.5), (0.5, 1.0)]:
    m = ContentBasedRecommender(EventFeaturizer(category_weight=cat_w, city_weight=city_w),
                                interest_strength=best1["interest_strength"], city_strength=best1["city_strength"]).fit(train, users, events)
    r = evaluate_ranking(m, task).mean(numeric_only=True)
    rows.append({"category_weight": cat_w, "city_weight": city_w, "recall@10": r["recall@10"], "ndcg@10": r["ndcg@10"]})
grid2 = pd.DataFrame(rows).sort_values("ndcg@10", ascending=False)
best2 = grid2.iloc[0]
grid2.round(4)

,category_weight,city_weight,recall@10,ndcg@10
1,0.50,0.25,0.7148,0.4431
3,1.00,0.50,0.7106,0.4422
2,0.50,0.50,0.6985,0.4354
0,0.25,0.50,0.6685,0.4202
4,0.50,1.00,0.6636,0.4172


The differences between settings are small (about ±0.02 NDCG), which is reassuring: the model is not fragile, and we are not winning by luck on one lucky setting.

## 5. Content-based rating prediction
To predict a **rating** we look at the user's own past ratings of **similar** events (item-kNN on content):

`prediction = b_u + Σ sim(e, j) × (r_j − b_u) / (Σ sim(e, j) + λ)`

- `b_u` = the user's average rating, **shrunk** toward the global average when they have few ratings: `(Σ r + 5·μ) / (n + 5)`.
- The sum runs over the 20 most similar events the user rated.
- `λ` (neighbour shrinkage) keeps the correction small when the similar events are only weakly similar.

In [11]:
rated_train = train.dropna(subset=["rating"])
rated_val = val.dropna(subset=["rating"])
print("ratings: train", len(rated_train), "| validation", len(rated_val))

rows = [{"model": "Global mean", **evaluate_rating(GlobalMeanPredictor().fit(rated_train), rated_val)}]
for lam in [0, 3, 10, 30]:
    cbf.neighbor_shrinkage = lam
    rows.append({"model": f"Content kNN (λ={lam})", "lambda": lam, **evaluate_rating(cbf, rated_val)})
rating_grid = pd.DataFrame(rows).round(4)
best_lambda = float(rating_grid.dropna(subset=["lambda"]).sort_values("RMSE").iloc[0]["lambda"])
cbf.neighbor_shrinkage = best_lambda
rating_grid

ratings: train 5056 | validation 1858


,model,MAE,RMSE,lambda
0,Global mean,0.8620,1.0223,NaN
1,Content kNN (λ=0),0.8758,1.0944,0.0
2,Content kNN (λ=3),0.8224,1.0009,3.0
3,Content kNN (λ=10),0.8235,0.9986,10.0
4,Content kNN (λ=30),0.8248,0.9987,30.0


**Reading the result:** a plain kNN (λ = 0) is *worse* than predicting the global mean — with only a handful of ratings per user, the neighbour correction mostly adds noise. Damping it fixes that and beats the baseline, but only slightly. Rating prediction is hard here because most users have few ratings and most events being rated are new. SVD (next notebook) adds **event biases** learned from other users.

## 6. Save the chosen settings

In [12]:
params = update_params({"cbf": {
    "interest_strength": float(best1["interest_strength"]), "city_strength": float(best1["city_strength"]),
    "category_weight": float(best2["category_weight"]), "city_weight": float(best2["city_weight"]),
    "neighbor_shrinkage": best_lambda,
}})
params["cbf"]

{'interest_strength': 5.0, 'city_strength': 1.0, 'category_weight': 0.5, 'city_weight': 0.25, 'neighbor_shrinkage': 10.0}

## 7. Cold start in action: a brand-new user
A user who just signed up, likes **Technology** and **Music**, lives in Alexandria, and has zero history.

In [13]:
new_users = pd.concat([users, pd.DataFrame([{"user_id": 99999, "age": 25, "city": "Alexandria",
                                              "interests": "Technology|Music", "signup_date": VAL_END}])])
cbf_new = ContentBasedRecommender().fit(train, new_users, events)
top = recommend(cbf_new, 99999, task.candidates, k=10)
events.set_index("event_id").loc[top, ["title", "category", "city", "price"]]

,title,category,city,price
event_id,,,,
497,Live Band Night,Music,Alexandria,220.0
145,Intro To Concert,Music,Alexandria,240.0
420,Oriental Gathering,Music,Alexandria,120.0
243,Pop Night,Music,Alexandria,110.0
201,Beats Night,Music,Alexandria,570.0
32,Dj Workshop,Music,Cairo,230.0
607,Pop Night,Music,Cairo,130.0
317,Stage Workshop,Music,Cairo,0.0
453,Cloud Meetup,Technology,Cairo,130.0


Every pick is Music or Technology, and the top five are in the user's own city — sensible with no history at all (there are few Technology events in this two-month window). Collaborative filtering could not produce anything for this user.

## Strengths and weaknesses
| ✔ Strengths | ✘ Weaknesses |
|---|---|
| Works for new events and new users | Over-specialisation: only "more of the same" |
| Explainable ("because you liked X") | Limited by what the text says |
| No other users' data needed | Cannot learn hidden tastes shared by similar people |

### Interview questions for this stage
- *How do you build a user profile in content-based filtering?* — A weighted average of the vectors of items the user engaged with; here weights reflect engagement strength and ratings, blended with stated interests.
- *Why cosine similarity rather than Euclidean distance?* — Cosine compares direction (topic mix), not length; a long description should not look different just because it has more words.
- *How do you handle new users?* — Their profile is built from the interests and city they gave at sign-up; behaviour takes over as history accumulates.

**Next:** `06_collaborative_svd.ipynb`.